We will add some new properties to the `entity` object to allow it to move about a plane to find food. With this addition, we can simulate a population growth contrained by limited available food resources. 

In [3]:
# Import required modules
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
from random import random, randint
from collections import Counter, namedtuple
from math import hypot

Collision detection:

2 objects
https://www.youtube.com/watch?v=XYzA_kPWyJ8
multiple objects
https://www.youtube.com/watch?v=789weryntzM


In [4]:
class Vector:
    """ A vector stores information and operations regarding the 
        direction and speed of motion.
    
        Parameters:
            vx (int or float): x component of Vector.
            vy (int or float): y component of Vector.
    """

    def __init__(self, vx, vy):
        self.vx = vx
        self.vy = vy

    def __repr__(self):
        return f"Vector({self.vx!r}, {self.vy!r})"

    def __abs__(self):
        """ Get the magnitude of the vector."""
        return hypot(self.vx, self.vy)

    def __bool__(self):
        """ Check truthiness of vector."""
        return bool(abs(self))

    def __add__(self, other):
        """ Add two vectors."""
        vx = self.vx + other.vx
        vy = self.vy + other.vy
        return Vector(vx, vy)

    def __mul__(self, scalar):
        """ Scale the vector."""
        return Vector(self.vx * scalar, self.vy * scalar)

        

In [5]:
Extent = namedtuple("area", ["width", "height"])

def trial(chance):
    """ Returns the boolean outcome of a trial given it's chance of success."""
    return random() <= chance

def collision(self, other):
    """ Detect whether two objects are touching."""
    dx = self.x - other.x
    dy = self.y - other.y
    dist = hypot(dx, dy)
    if dist <= self.radius + other.radius:
        return True
    return False

class Entity():
    """ The entity object represents a live entity.

        Parameters:
            x (int or float): x location of Entity.
            y (int or float): y location of Entity.
    """
    
    def __init__(self, x, y, radius, motion):
        self.x = x
        self.y = y
        self.radius = radius
        self.motion = motion
        self.period = 0
    
    def __repr__(self):
        return str(self.period)

    def __iadd__(self, other):
        """  Update the location of self by Vector other."""
        self.x = self.x + other.vx
        self.y = self.y + other.vy
        return self
    
    def __call__(self):
        """ advance the entity by one period."""
        self.period += 1
        self += self.motion
        return self

class Environment():
    """ The environment tracks the population and parameters of the simulation.

        Parameters:
            repro_chance (float):               Chance of reproduction
            death_chance (float):               Chance of death
            starting_pop (int):                 Starting population
            extent (iterable [width, height]):  Extent of habitat
    """

    def __init__(self, repro_chance, death_chance, starting_pop, extent):
        self.repro_chance = repro_chance
        self.death_chance = death_chance
        self.starting_pop = starting_pop
        self.extent = Extent(*extent)

        self.population = [
            Entity(randint(0, self.extent.width), randint(0, self.extent.height))
            for p in range(0, starting_pop)
        ]
        self.history = [starting_pop]
        self.mortuary = Counter()

    def __call__(self):
        """ Advance the simulation by one period."""
        # Simulate period
        survive, expire, births = [], [], []
        # First determine which entities expired by end of previous period.
        for e in self.population:
            if trial(self.death_chance):
               expire.append(e().period) 
            else:
                survive.append(e())
                # Determine if the entity reproduced by end of current period.
                if trial(self.repro_chance):
                    births.append(Entity(randint(0, self.extent.width), randint(0, self.extent.height)))

        # Finally, update attributes
        self.population = survive + births
        self.mortuary.update(expire)
        self.history.append(len(self.population))



In [6]:
e1 = Entity(3, 3, .5, Vector(.5, .5))
e2 = Entity(6, 6, .5, Vector(-.5, -.5))


In [12]:

e1()
print(e1, e1.x, e1.y)
e2()
print(e2, e2.x, e2.y)
print(collision(e1, e2))

6 6.0 6.0
6 3.0 3.0
False
